# Building Analytics Marts

Rebuilds the core analytics marts from the processed fact table (`final.csv`). These marts support the first three business themes: revenue analysis, user behavior understanding, and marketing optimization. Cashback simulation is kept as an optional supporting mart.

| Mart | Grain | Purpose |
|---|---|---|
| `mart_revenue` | 1 row = 1 month | Monthly financial and user KPIs (GMV, revenue, AOV, ARPU) |
| `mart_carrier` | 1 row = month x carrier | Carrier-level performance, transaction share, and take rate (`revenue_share` formula still needs correction) |
| `mart_behavior` | 1 row = month x Type_user x Purchase_status | Purchase type contribution analysis (Buying for others vs. Standard) and trends |
| `mart_demographic` | 1 row = 1 user | Detailed user segments (Age, Gender, Location, Lifecycle, Behavior) at user grain |
| `mart_cohort` | 1 row = cohort_month x period | Retention analysis and cohort performance |
| `mart_churn` | 1 row = lifecycle_stage x recency_band x behavior_segment | Churn snapshot, risk distribution, and reactivation sizing |
| `mart_timing` | 1 row = weekday x day_of_month | Peak usage timing and day-of-month trends |
| `mart_simulation` | 1 row = 1 carrier | Optional what-if analysis for cashback policy changes |

## 1. Setup & Load

In [1]:
import pandas as pd
import os
from pathlib import Path

In [2]:
BASE_DIR = Path.cwd().parent
SRC_PATH   = BASE_DIR / "Data" / "processed" / "final.csv"
MART_DIR   = BASE_DIR / "Data" / "mart"
MART_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
df = pd.read_csv(SRC_PATH)
df['Date']           = pd.to_datetime(df['Date'])
df['First_tran_date']= pd.to_datetime(df['First_tran_date'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13494 entries, 0 to 13493
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   user_id          13494 non-null  int64         
 1   order_id         13494 non-null  int64         
 2   Date             13494 non-null  datetime64[ns]
 3   Amount           13494 non-null  int64         
 4   Merchant_id      13494 non-null  int64         
 5   Purchase_status  13494 non-null  int64         
 6   Merchant_name    13494 non-null  object        
 7   Rate_pct         13494 non-null  int64         
 8   Revenue          13494 non-null  float64       
 9   First_tran_date  13494 non-null  datetime64[ns]
 10  Location         13494 non-null  object        
 11  Age              13494 non-null  object        
 12  Gender           13494 non-null  object        
 13  Type_user        13494 non-null  object        
dtypes: datetime64[ns](2), float64(1), int6

## 2. Derived Columns


### Time Features

In [4]:
# month as 'YYYY-MM' string (sortable)
df['month']        = df['Date'].dt.to_period('M').astype(str)

# cohort month from first transaction
df['cohort_month'] = df['First_tran_date'].dt.to_period('M').astype(str)

# cohort year from first transaction
df['cohort_year']  = df['First_tran_date'].dt.year

# Weekday (0 = Monday, 6 = Sunday)
df['weekday'] = df['Date'].dt.weekday

# Weekend flag (1 = weekend, 0 = weekday)
df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)

# Day of month (1–31)
df['day_of_month'] = df['Date'].dt.day

print(df[['month','cohort_month','cohort_year','weekday','is_weekend','day_of_month']].head())

     month cohort_month  cohort_year  weekday  is_weekend  day_of_month
0  2020-01      2018-12         2018        2           0             1
1  2020-01      2019-12         2019        2           0             1
2  2020-01      2019-11         2019        2           0             1
3  2020-01      2019-10         2019        2           0             1
4  2020-01      2019-12         2019        2           0             1


### Business Features

In [5]:
# carrier alias (Merchant_name is the carrier)
df['carrier'] = df['Merchant_name']

# human-readable purchase label
df['purchase_label'] = df['Purchase_status'].map({1: 'Purchase for others', 0: 'Standard'})

print(df[['carrier','purchase_label']].head())

    carrier purchase_label
0  Mobifone           Khác
1  Mobifone           Khác
2   Viettel           Khác
3  Mobifone           Khác
4   Viettel           Khác


---
## 3. `mart_revenue`
**Grain:** 1 row = 1 month  
Monthly GMV, revenue, user counts and growth rates.

In [7]:
mart_revenue = (
    df.groupby('month', sort=True)
      .agg(
          total_txn    = ('order_id',  'count'),
          total_amount = ('Amount',    'sum'),
          revenue      = ('Revenue',   'sum'),
          active_users = ('user_id',   'nunique'),
          new_users    = ('user_id',   lambda x: x[df.loc[x.index, 'Type_user'] == 'New'].nunique()),
      )
      .reset_index()
)

In [8]:
# ── derived metrics ───────────────────────────────────────────────────────────
mart_revenue['returning_users']     = mart_revenue['active_users'] - mart_revenue['new_users']
mart_revenue['AOV']                 = mart_revenue['total_amount'] / mart_revenue['total_txn']
mart_revenue['ARPU']    = mart_revenue['revenue']      / mart_revenue['active_users']
mart_revenue['take_rate']           = mart_revenue['revenue']      / mart_revenue['total_amount']
mart_revenue['pct_new_users']       = mart_revenue['new_users']    / mart_revenue['active_users']
# MoM growth (pct_change on sorted month)
mart_revenue['mom_gmv_growth']      = mart_revenue['total_amount'].pct_change()
mart_revenue['mom_revenue_growth']  = mart_revenue['revenue'].pct_change()

mart_revenue

,month,total_txn,total_amount,revenue,active_users,new_users,returning_users,AOV,ARPU,take_rate,pct_new_users,mom_gmv_growth,mom_revenue_growth
0,2020-01,999,53520000,1401000.0,999,85,914,53573.573574,1402.402402,0.026177,0.085085,NaN,NaN
1,2020-02,972,52680000,1378500.0,969,91,878,54197.530864,1422.600619,0.026167,0.093911,-0.015695,-0.016060
2,2020-03,1098,58200000,1584000.0,1098,113,985,53005.464481,1442.622951,0.027216,0.102914,0.104784,0.149075
3,2020-04,1027,54150000,1488300.0,1027,81,946,52726.387537,1449.172347,0.027485,0.078870,-0.069588,-0.060417
4,2020-05,1104,52950000,1463700.0,1104,72,1032,47961.956522,1325.815217,0.027643,0.065217,-0.022161,-0.016529
5,2020-06,1118,58250000,1617200.0,1116,62,1054,52101.967800,1449.103943,0.027763,0.055556,0.100094,0.104871
6,2020-07,1153,58940000,1581900.0,1149,72,1077,51118.820468,1376.762402,0.026839,0.062663,0.011845,-0.021828
7,2020-08,1153,59950000,1618700.0,1153,84,1069,51994.796184,1403.902862,0.027001,0.072853,0.017136,0.023263
8,2020-09,1189,63400000,1702200.0,1189,82,1107,53322.119428,1431.623213,0.026849,0.068966,0.057548,0.051585
9,2020-10,1211,61690000,1690900.0,1209,79,1130,50941.370768,1398.593879,0.027410,0.065343,-0.026972,-0.006638


---
## 4. `mart_amount_distribution`
**Grain:** 1 row = 1 month

In [10]:
mart_amount_distribution = (
    df.groupby(['month','Amount'],sort = True,observed=True)
        .agg(
            txn_count   = ('order_id', 'count'),
            user_count  = ('user_id', 'nunique'),
            total_amount= ('Amount', 'sum'),
            revenue     = ('Revenue', 'sum')
        )
    .reset_index()
)

**pct_txn**

Percentage of transactions for each amount level within a given month.

This shows how transaction volume is distributed across different top-up amounts, helping identify the most frequently used amounts by users.

In [21]:
mart_amount_distribution['pct_txn'] = (
    mart_amount_distribution['txn_count'] /
    mart_amount_distribution.groupby('month')['txn_count'].transform('sum')
)

**pct_revenue**

Percentage of total revenue contributed by each amount level within a given month.

This helps identify which top-up amounts drive the most revenue and whether revenue is concentrated in higher or lower value transactions.

In [22]:
mart_amount_distribution['pct_revenue'] = ( 
    mart_amount_distribution['revenue'] / 
    mart_amount_distribution.groupby('month')['revenue'].transform('sum') 
    )

**rank_txn**

Ranking of amount levels based on transaction count within each month (1 = highest).

This is used to identify the most popular top-up amounts and enables easy filtering for top N amounts (e.g., top 5).

In [23]:
mart_amount_distribution['rank_txn'] = ( 
    mart_amount_distribution .groupby('month')['txn_count'] .rank(method='dense', ascending=False) )

**rank_revenue**

Ranking of amount levels based on revenue contribution within each month (1 = highest).

This helps highlight which amounts contribute the most to revenue and supports top N analysis for revenue concentration.

In [24]:
mart_amount_distribution['rank_revenue'] = (
    mart_amount_distribution
        .groupby('month')['revenue']
        .rank(method='dense', ascending=False)
)

**Top 5 rank revenue**

In [27]:
top5_txn = mart_amount_distribution[
    mart_amount_distribution['rank_txn'] <= 1
]
top5_txn

,month,Amount,txn_count,user_count,total_amount,revenue,pct_txn,pct_revenue,rank_txn,rank_revenue
1,2020-01,20000,260,260,5200000,140800.0,0.260260,0.100500,1.0,4.0
10,2020-02,20000,249,249,4980000,138200.0,0.256173,0.100254,1.0,4.0
21,2020-03,10000,265,265,2650000,73300.0,0.241348,0.046275,1.0,7.0
35,2020-04,50000,249,249,12450000,347000.0,0.242454,0.233152,1.0,2.0
42,2020-05,10000,286,286,2860000,80400.0,0.259058,0.054929,1.0,6.0
55,2020-06,50000,285,285,14250000,389000.0,0.254919,0.240539,1.0,2.0
63,2020-07,10000,291,291,2910000,78500.0,0.252385,0.049624,1.0,7.0
73,2020-08,10000,279,279,2790000,75200.0,0.241977,0.046457,1.0,6.0
83,2020-09,10000,293,293,2930000,83000.0,0.246426,0.048760,1.0,6.0
95,2020-10,20000,330,330,6600000,171600.0,0.272502,0.101484,1.0,4.0


---
## 4. `mart_carrier`
**Grain:** 1 row = month x carrier  
Tracks carrier-level GMV, revenue, users, transaction share, and take rate.  

**Note:** `txn_share` is correct, but the current `revenue_share` column is divided by monthly `total_amount` instead of monthly `revenue`, so the metric is mislabeled and should be fixed in code.

In [ ]:
mart_carrier = (
    df.groupby(['month', 'carrier'], sort=True, observed=True)
      .agg(
          total_txn    = ('order_id', 'count'),
          total_amount = ('Amount',   'sum'),
          revenue      = ('Revenue',  'sum'),
          users        = ('user_id',  'nunique'),
      )
      .reset_index()
)

In [ ]:
# ── per-carrier derived metrics ───────────────────────────────────────────────
mart_carrier['AOV']       = mart_carrier['total_amount'] / mart_carrier['total_txn']
mart_carrier['take_rate'] = mart_carrier['revenue']      / mart_carrier['total_amount']

# ── merge monthly totals from mart_revenue to compute shares ─────────────────
monthly_totals = mart_revenue[['month', 'total_txn', 'total_amount']].rename(
    columns={'total_txn': 'month_total_txn', 'total_amount': 'month_total_amount'}
)
mart_carrier = mart_carrier.merge(monthly_totals, on='month', how='left')

mart_carrier['txn_share']     = mart_carrier['total_txn']    / mart_carrier['month_total_txn']
mart_carrier['revenue_share'] = mart_carrier['revenue']      / mart_carrier['month_total_amount']

# drop temp helper cols
mart_carrier.drop(columns=['month_total_txn', 'month_total_amount'], inplace=True)

mart_carrier.head(10)

,month,carrier,total_txn,total_amount,revenue,users,AOV,take_rate,txn_share,revenue_share
0,2020-01,Mobifone,282,15214234.0,456427.02,282,53951.184397,0.03,0.282000,0.008482
1,2020-01,Vietnamobile,42,1140000.0,45600.00,42,27142.857143,0.04,0.042000,0.000847
2,2020-01,Viettel,513,29530000.0,590600.00,513,57563.352827,0.02,0.513000,0.010975
3,2020-01,Vinaphone,163,7930000.0,317200.00,163,48650.306748,0.04,0.163000,0.005894
4,2020-02,Mobifone,280,13570000.0,407100.00,278,48464.285714,0.03,0.288066,0.007728
5,2020-02,Others,1,50000.0,2000.00,1,50000.000000,0.04,0.001029,0.000038
6,2020-02,Vietnamobile,30,760000.0,30400.00,30,25333.333333,0.04,0.030864,0.000577
7,2020-02,Viettel,480,29650000.0,593000.00,480,61770.833333,0.02,0.493827,0.011257
8,2020-02,Vinaphone,181,8650000.0,346000.00,181,47790.055249,0.04,0.186214,0.006568
9,2020-03,Mobifone,320,17120000.0,513600.00,320,53500.000000,0.03,0.291439,0.008825


---
## 5. `mart_behavior`
**Grain:** 1 row = month x Type_user x Purchase_status  
Answers: Does 'Buying for others' have higher AOV? How much revenue and transaction volume does each purchase type contribute?  

**Gap:** the mart does not yet include a user-share metric such as `pct_users`, so it cannot fully answer whether new users buy for others more often at the user level.

In [12]:
mart_behavior = (
    df.groupby(['month', 'Type_user', 'purchase_label'], sort=True, observed=True)
      .agg(
          users   = ('user_id',  'nunique'),
          txn     = ('order_id', 'count'),
          amount  = ('Amount',   'sum'),
          revenue = ('Revenue',  'sum'),
      )
      .reset_index()
      .rename(columns={'purchase_label': 'Purchase_status'})
)

mart_behavior['AOV'] = mart_behavior['amount'] / mart_behavior['txn']

mart_behavior.head(12)

,month,Type_user,Purchase_status,users,txn,amount,revenue,AOV
0,2020-01,Current,Khác,797,797,36120000.0,925200.00,45319.949812
1,2020-01,Current,Mua hộ,118,118,14014234.0,386927.02,118764.694915
2,2020-01,New,Khác,79,79,2560000.0,67100.00,32405.063291
3,2020-01,New,Mua hộ,6,6,1120000.0,30600.00,186666.666667
4,2020-02,Current,Khác,758,760,34660000.0,891700.00,45605.263158
5,2020-02,Current,Mua hộ,120,121,14140000.0,391300.00,116859.504132
6,2020-02,New,Khác,84,84,3140000.0,78100.00,37380.952381
7,2020-02,New,Mua hộ,7,7,740000.0,17400.00,105714.285714
8,2020-03,Current,Khác,840,840,36050000.0,964800.00,42916.666667
9,2020-03,Current,Mua hộ,145,145,17660000.0,508100.00,121793.103448


### `mart_behavior` — % Share Metrics
Calculating the percentage of revenue and transaction share for each segment (Purchase type × User type) per month. This helps answer: **"How much does 'Purchase for others' contribute to total revenue?"**

In [ ]:
# % share of revenue and txn per purchase_type within each month
monthly_rev = mart_behavior.groupby('month')['revenue'].transform('sum')
monthly_txn = mart_behavior.groupby('month')['txn'].transform('sum')
mart_behavior['pct_revenue'] = mart_behavior['revenue'] / monthly_rev
mart_behavior['pct_txn']     = mart_behavior['txn']     / monthly_txn

mart_behavior.head(8)

---
## 6. `mart_demographic`
**Grain:** 1 row = 1 user  
User-level summary. Slice by Age, Gender, Location, and Cohort inside Power BI via relationships. There is no month dimension here to avoid combinatorial explosion.  

**Strength:** good for segmentation, lifecycle flags, and identifying upsell candidates at user grain.  
**Gap:** this mart does not yet provide aggregated churn rate, demographic revenue share, Pareto concentration, or reactivation scoring directly.

In [14]:
def safe_mode(s):
    """Return the first mode value (most frequent carrier per user)."""
    m = s.mode()
    return m.iloc[0] if not m.empty else None

mart_demographic = (
    df.groupby('user_id', sort=False)
      .agg(
          Age           = ('Age',           'first'),
          Gender        = ('Gender',         'first'),
          Location      = ('Location',       'first'),
          cohort_year   = ('cohort_year',    'first'),
          First_tran_date = ('First_tran_date', 'first'),
          total_txn     = ('order_id',       'count'),
          total_amount  = ('Amount',         'sum'),
          revenue       = ('Revenue',        'sum'),
          active_months = ('month',          'nunique'),
          carrier_mode  = ('carrier',        safe_mode),
      )
      .reset_index()
)

mart_demographic['AOV'] = mart_demographic['total_amount'] / mart_demographic['total_txn']

print(mart_demographic.shape)
mart_demographic.head()

(13390, 12)


,user_id,Age,Gender,Location,cohort_year,First_tran_date,total_txn,total_amount,revenue,active_months,carrier_mode,AOV
0,21269588,>37,FEMALE,HN,2018,2018-12-11,1,10000.0,300.0,1,Mobifone,10000.0
1,28097592,>37,FEMALE,HN,2019,2019-12-30,1,20000.0,600.0,1,Mobifone,20000.0
2,47435144,18_to_22,FEMALE,HN,2019,2019-11-11,1,10000.0,200.0,1,Viettel,10000.0
3,29080935,18_to_22,FEMALE,HN,2019,2019-10-24,1,10000.0,300.0,1,Mobifone,10000.0
4,14591075,18_to_22,FEMALE,Other Cities,2019,2019-12-28,1,10000.0,200.0,1,Viettel,10000.0


### `mart_demographic` - Lifecycle and Behavior Segmentation

**Lifecycle Stage** - Determined based on `days_since_last_txn` relative to the dataset end date.

**Behavior Segment** - Segments users into four quadrants based on transaction frequency and AOV:  
- `High Freq Low Value` -> Primary target (frequent top-ups, small amounts)  
- `High Value Low Freq` -> Upsell candidates  
- `High Freq High Value` -> Champions (power users)  
- `Low Freq Low Value` -> Dormant or low-priority users

In [ ]:
import numpy as np

# ── last transaction date & recency ───────────────────────────────────────────
DATA_END_DATE = df['Date'].max()

last_txn = (
    df.groupby('user_id')['Date']
      .max()
      .rename('last_txn_date')
      .reset_index()
)
mart_demographic = mart_demographic.merge(last_txn, on='user_id', how='left')
mart_demographic['days_since_last_txn'] = (
    DATA_END_DATE - mart_demographic['last_txn_date']
).dt.days
mart_demographic['recency_band'] = pd.cut(
    mart_demographic['days_since_last_txn'],
    bins=[-1, 30, 60, 90, 180, float('inf')],
    labels=['0-30 days', '31-60 days', '61-90 days', '91-180 days', '180+ days']
)

# ── churn flag (inactive > 90 days) ───────────────────────────────────────────
mart_demographic['churn_flag'] = (
    mart_demographic['days_since_last_txn'] > 90
).astype(int)

# ── lifecycle stage ────────────────────────────────────────────────────────────
def lifecycle_stage(row):
    if row['total_txn'] == 1 and row['days_since_last_txn'] <= 30:
        return 'New'
    elif row['days_since_last_txn'] > 90:
        return 'Churned'
    elif row['days_since_last_txn'] > 60:
        return 'At Risk'
    else:
        return 'Active'

mart_demographic['lifecycle_stage'] = mart_demographic.apply(lifecycle_stage, axis=1)

# ── frequency metric ──────────────────────────────────────────────────────────
mart_demographic['avg_txn_per_active_month'] = (
    mart_demographic['total_txn'] / mart_demographic['active_months']
)

# ── behavior segment (2×2 quadrant) ───────────────────────────────────────────
freq_median = mart_demographic['avg_txn_per_active_month'].median()
aov_median  = mart_demographic['AOV'].median()

def behavior_seg(row):
    high_freq  = row['avg_txn_per_active_month'] >= freq_median
    high_value = row['AOV'] >= aov_median
    if   high_freq and not high_value: return 'High Freq Low Value'   # primary target
    elif high_value and not high_freq: return 'High Value Low Freq'   # upsell
    elif high_freq and high_value:     return 'High Freq High Value'  # champions
    else:                              return 'Low Freq Low Value'    # dormant

mart_demographic['behavior_segment'] = mart_demographic.apply(behavior_seg, axis=1)
mart_demographic['upsell_flag'] = (
    mart_demographic['behavior_segment'] == 'High Value Low Freq'
).astype(int)

print('Lifecycle distribution:')
print(mart_demographic['lifecycle_stage'].value_counts())
print('\nBehavior segment distribution:')
print(mart_demographic['behavior_segment'].value_counts())

mart_demographic.head()

---
## 8. `mart_churn`
**Grain:** 1 row = lifecycle_stage x recency_band x behavior_segment  
Snapshot mart for churn monitoring and reactivation planning. It shows how many users are active, at risk, or churned, where revenue is concentrated, and which segments are worth prioritizing for reactivation.

In [ ]:
mart_churn = (
    mart_demographic
      .groupby(['lifecycle_stage', 'recency_band', 'behavior_segment'], observed=True)
      .agg(
          users=('user_id', 'nunique'),
          total_txn=('total_txn', 'sum'),
          total_amount=('total_amount', 'sum'),
          revenue=('revenue', 'sum'),
          avg_days_since_last_txn=('days_since_last_txn', 'mean'),
          avg_aov=('AOV', 'mean'),
          avg_txn_per_active_month=('avg_txn_per_active_month', 'mean'),
          churn_rate=('churn_flag', 'mean'),
          upsell_candidates=('upsell_flag', 'sum')
      )
      .reset_index()
)

mart_churn['user_share'] = mart_churn['users'] / mart_churn['users'].sum()
mart_churn['revenue_share'] = mart_churn['revenue'] / mart_churn['revenue'].sum()
mart_churn['AOV'] = mart_churn['total_amount'] / mart_churn['total_txn']
mart_churn['ARPU'] = mart_churn['revenue'] / mart_churn['users']

stage_order = ['New', 'Active', 'At Risk', 'Churned']
recency_order = ['0-30 days', '31-60 days', '61-90 days', '91-180 days', '180+ days']
behavior_order = [
    'High Freq High Value',
    'High Freq Low Value',
    'High Value Low Freq',
    'Low Freq Low Value'
]

mart_churn['lifecycle_stage'] = pd.Categorical(mart_churn['lifecycle_stage'], categories=stage_order, ordered=True)
mart_churn['recency_band'] = pd.Categorical(mart_churn['recency_band'], categories=recency_order, ordered=True)
mart_churn['behavior_segment'] = pd.Categorical(mart_churn['behavior_segment'], categories=behavior_order, ordered=True)
mart_churn = mart_churn.sort_values(['lifecycle_stage', 'recency_band', 'revenue'], ascending=[True, True, False]).reset_index(drop=True)

print(mart_churn.shape)
mart_churn.head(12)

---
## 9. `mart_cohort`
**Grain:** 1 row = cohort_month x period_number  
Answers: Do users return after their first month? Which cohorts have the best retention?

- `period_number = 0` is the month the user first joined.
- `retention_rate = retained_users / cohort_size`

In [ ]:
# ── month × user activity ─────────────────────────────────────────────────────
user_months = (
    df.groupby(['user_id', 'month'])
      .size()
      .reset_index(name='txn_count')
)
user_cohort = (
    df.groupby('user_id')['cohort_month']
      .first()
      .reset_index()
)
cohort_df = user_months.merge(user_cohort, on='user_id')

# ── period offset (months since first transaction) ────────────────────────────
cohort_df['cohort_p'] = pd.PeriodIndex(cohort_df['cohort_month'], freq='M')
cohort_df['month_p']  = pd.PeriodIndex(cohort_df['month'], freq='M')
cohort_df['period_number'] = (
    cohort_df['month_p'] - cohort_df['cohort_p']
).apply(lambda x: x.n)

# ── cohort size at period 0 ───────────────────────────────────────────────────
cohort_size = (
    cohort_df[cohort_df['period_number'] == 0]
    .groupby('cohort_month')['user_id']
    .nunique()
    .rename('cohort_size')
    .reset_index()
)

# ── retained users per period ─────────────────────────────────────────────────
mart_cohort = (
    cohort_df.groupby(['cohort_month', 'period_number'])['user_id']
    .nunique()
    .rename('retained_users')
    .reset_index()
    .merge(cohort_size, on='cohort_month', how='left')
)
mart_cohort['retention_rate'] = (
    mart_cohort['retained_users'] / mart_cohort['cohort_size']
)

print(mart_cohort.shape)
mart_cohort.head(15)

---
## 10. `mart_simulation`
**Grain:** 1 row = 1 carrier  

Optional what-if mart: if MoMo raises cashback from 1% to the proposed rates, how much extra cost is created and how much additional volume is needed to break even?

```
current_cashback_rate = 1%  (all carriers)
proposed_rates        = Viettel 2%, Mobifone 2.5%, Vinaphone/Vietnamobile/Gmobile 3%

commission_revenue    = Amount x Rate_pct/100   (from carrier to MoMo)
cashback_cost         = Amount x cashback_rate  (from MoMo to user)
net_revenue           = commission_revenue - cashback_cost

delta_cashback        = proposed_cashback - current_cashback
delta_revenue         = -delta_cashback
breakeven_volume_increase = delta_cashback / net_commission_rate
```

In [ ]:
# ── carrier-level totals
carrier_totals = (
    df.groupby('carrier', observed=True)
      .agg(
          total_amount     = ('Amount',  'sum'),
          current_revenue  = ('Revenue', 'sum'),   # Amount × Rate_pct/100
      )
      .reset_index()
)

# ── commission rate per carrier (Rate_pct / 100) ──────────────────────────────
commission_rate = (
    df.groupby('carrier', observed=True)['Rate_pct']
      .first() / 100
).rename('commission_rate')
carrier_totals = carrier_totals.merge(commission_rate.reset_index(), on='carrier')

# ── cashback policy config ────────────────────────────────────────────────────
current_cashback_rate = 0.01

proposed_rates = {
    'Viettel':      0.02,
    'Mobifone':     0.025,
    'Vinaphone':    0.03,
    'Vietnamobile': 0.03,
    'Gmobile':      0.03,
}

carrier_totals['current_rate']   = current_cashback_rate
carrier_totals['proposed_rate']  = carrier_totals['carrier'].map(proposed_rates)

# ── simulation calculations ───────────────────────────────────────────────────
carrier_totals['current_cashback']  = carrier_totals['total_amount'] * carrier_totals['current_rate']
carrier_totals['proposed_cashback'] = carrier_totals['total_amount'] * carrier_totals['proposed_rate']

carrier_totals['delta_cashback']    = carrier_totals['proposed_cashback'] - carrier_totals['current_cashback']
carrier_totals['delta_revenue']     = -carrier_totals['delta_cashback']   # MoMo bears this cost

# Net commission rate = commission_rate - current_cashback_rate
# breakeven: how much extra volume needed to restore current_revenue
# new_volume × net_rate = current_revenue  →  extra_volume = delta_cashback / net_rate
carrier_totals['net_commission_rate']       = carrier_totals['commission_rate'] - current_cashback_rate
carrier_totals['breakeven_volume_increase'] = carrier_totals['delta_cashback'] / carrier_totals['net_commission_rate']

# ── final column order ────────────────────────────────────────────────────────
mart_simulation = carrier_totals[[
    'carrier', 'current_rate', 'proposed_rate',
    'total_amount', 'current_revenue',
    'current_cashback', 'proposed_cashback',
    'delta_cashback', 'delta_revenue',
    'net_commission_rate', 'breakeven_volume_increase'
]]

mart_simulation

,carrier,current_rate,proposed_rate,total_amount,current_revenue,current_cashback,proposed_cashback,delta_cashback,delta_revenue,net_commission_rate,breakeven_volume_increase
0,Gmobile,0.01,0.030,80000.0,3200.00,800.00,2400.00,1600.00,-1600.00,0.03,5.333333e+04
1,Mobifone,0.01,0.025,191904234.0,5757127.02,1919042.34,4797605.85,2878563.51,-2878563.51,0.02,1.439282e+08
2,Vietnamobile,0.01,0.030,21680000.0,867200.00,216800.00,650400.00,433600.00,-433600.00,0.03,1.445333e+07
3,Viettel,0.01,0.020,357620000.0,7152400.00,3576200.00,7152400.00,3576200.00,-3576200.00,0.01,3.576200e+08
4,Vinaphone,0.01,0.030,123320000.0,4932800.00,1233200.00,3699600.00,2466400.00,-2466400.00,0.03,8.221333e+07


---
## 11. `mart_timing`
**Grain:** 1 row = weekday x day_of_month  
Answers: At what times and on which days do users tend to top up the most?

In [ ]:
WEEKDAY_MAP = {0:'Mon', 1:'Tue', 2:'Wed', 3:'Thu', 4:'Fri', 5:'Sat', 6:'Sun'}

mart_timing = (
    df.groupby(['weekday', 'is_weekend', 'day_of_month'], observed=True)
      .agg(
          txn_count    = ('order_id', 'count'),
          total_amount = ('Amount',   'sum'),
          revenue      = ('Revenue',  'sum'),
          unique_users = ('user_id',  'nunique'),
      )
      .reset_index()
)
mart_timing['avg_txn_per_user'] = (
    mart_timing['txn_count'] / mart_timing['unique_users']
)
mart_timing['weekday_name'] = mart_timing['weekday'].map(WEEKDAY_MAP)

print(mart_timing.shape)
mart_timing.head(10)

---

## 12. Output Analytics Marts
Exports the current core marts used for revenue analysis, user behavior analysis, and marketing optimization.

In [ ]:
mart_revenue.to_csv(MART_DIR / 'mart_revenue.csv',              index=False, encoding='utf-8-sig')
mart_carrier.to_csv(MART_DIR / 'mart_carrier.csv',              index=False, encoding='utf-8-sig')
mart_behavior.to_csv(MART_DIR / 'mart_behavior.csv',            index=False, encoding='utf-8-sig')
mart_demographic.to_csv(MART_DIR / 'mart_demographic.csv',      index=False, encoding='utf-8-sig')
mart_churn.to_csv(MART_DIR / 'mart_churn.csv',                  index=False, encoding='utf-8-sig')
#mart_simulation.to_csv(MART_DIR / 'mart_simulation.csv',        index=False, encoding='utf-8-sig')
mart_cohort.to_csv(MART_DIR / 'mart_cohort.csv',                index=False, encoding='utf-8-sig')
mart_timing.to_csv(MART_DIR / 'mart_timing.csv',                index=False, encoding='utf-8-sig')
mart_amount_distribution.to_csv(
    MART_DIR / 'mart_amount_distribution.csv', index=False, encoding='utf-8-sig'
)
print('Core analytics marts exported successfully.')


---
## 13. Summary
The table below maps each exported mart to its grain, key metrics, and the business questions it currently supports.

| File | Grain | Key Metrics | Answers |
|---|---|---|---|
| `mart_revenue.csv` | month | `total_txn`, `total_amount`, `revenue`, `active_users`, `new_users`, `returning_users`, `AOV`, `ARPU`, `take_rate`, `pct_new_users`, MoM growth | 1.1, 1.2, 1.5 |
| `mart_carrier.csv` | month x carrier | `total_txn`, `total_amount`, `revenue`, `users`, `AOV`, `take_rate`, `txn_share`; `revenue_share` exists but is currently mislabeled | 1.3 (partial) |
| `mart_behavior.csv` | month x Type_user x Purchase_status | `users`, `txn`, `amount`, `revenue`, `AOV`, `pct_revenue`, `pct_txn` | 1.6 (partial for user-share question) |
| `mart_amount_distribution.csv` | month x Amount | `txn_count`, `user_count`, `total_amount`, `revenue`, `pct_txn`, `pct_revenue`, `rank_txn`, `rank_revenue` | 1.4 |
| `mart_demographic.csv` | user | `total_txn`, `total_amount`, `revenue`, `active_months`, `AOV`, `days_since_last_txn`, `recency_band`, `churn_flag`, `lifecycle_stage`, `avg_txn_per_active_month`, `behavior_segment`, `upsell_flag` | 2.1, 2.4, 2.5, 2.6, 3.1, 3.3 (partial) |
| `mart_churn.csv` | lifecycle_stage x recency_band x behavior_segment | `users`, `user_share`, `revenue`, `revenue_share`, `churn_rate`, `avg_days_since_last_txn`, `avg_txn_per_active_month`, `upsell_candidates` | 2.3, 2.6, 3.1, 3.2 |
| `mart_cohort.csv` | cohort_month x period | `retained_users`, `cohort_size`, `retention_rate` | 2.2 |
| `mart_timing.csv` | weekday x day_of_month | `txn_count`, `total_amount`, `revenue`, `unique_users`, `avg_txn_per_user` | 3.4 |
| `mart_simulation.csv` | carrier | `delta_revenue`, `breakeven_volume_increase` | 4.x optional; deferred for now |

In [ ]:
for name, frame in {
    'mart_revenue':              mart_revenue,
    'mart_carrier':              mart_carrier,
    'mart_behavior':             mart_behavior,
    'mart_amount_distribution':  mart_amount_distribution,
    'mart_demographic':         mart_demographic,
    'mart_churn':               mart_churn,
    'mart_cohort':               mart_cohort,
    'mart_timing':               mart_timing,
    #'mart_simulation':           mart_simulation,
}.items():
    print(f"{name:30s}  {frame.shape[0]:>6} rows × {frame.shape[1]:>2} cols")